# 04 — Interpretabilidade do Modelo

**Modelo final:** HistGradientBoosting — Cenário B (29 features, ordinal + contínuo) — threshold = 0.40

Este notebook responde três perguntas:

1. **O que o modelo aprendeu?** — quais features mais impactam a previsão de prematuridade (SHAP global)
2. **Por que o modelo previu isso para este paciente?** — decomposição da previsão individual (SHAP local)
3. **Onde o modelo falha e para quem?** — análise estratificada por grupo demográfico

---

**Contexto clínico:** o modelo é uma ferramenta de *triagem populacional* — não diagnóstico. Ao threshold 0.40, identifica ~81% dos prematuros a um custo de ~7 falsos positivos para cada verdadeiro positivo.

In [ ]:
import sys
import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import shap
from sklearn.calibration import calibration_curve
from sklearn.metrics import recall_score, precision_score, fbeta_score

sys.path.insert(0, "../src")
from pipeline import apply_scenario

warnings.filterwarnings("ignore")

RANDOM_STATE  = 42
SHAP_SAMPLE   = 2_000
FIGURES_DIR   = Path("../results/figures")
METRICS_DIR   = Path("../results/metrics")
ARTIFACTS_DIR = Path("../results/artifacts")

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 130, "font.size": 11})

In [ ]:
X_train = pd.read_parquet("../data/X_train.parquet")
X_test  = pd.read_parquet("../data/X_test.parquet")
y_train = pd.read_parquet("../data/y_train.parquet").squeeze()
y_test  = pd.read_parquet("../data/y_test.parquet").squeeze()

XB_train = apply_scenario(X_train, "B")
XB_test  = apply_scenario(X_test,  "B")

with open(ARTIFACTS_DIR / "best_model_calibrated.pkl", "rb") as f:
    model = pickle.load(f)

with open(METRICS_DIR / "best_model_operational_metrics.json") as f:
    ops = json.load(f)

THRESHOLD = ops["threshold"]
y_prob    = model.predict_proba(XB_test)[:, 1]
y_pred    = (y_prob >= THRESHOLD).astype(int)

print(f"Modelo   : {ops['model']} | Cenário: {ops['scenario']} | Threshold: {THRESHOLD:.2f}")
print(f"Test set : {len(y_test):,} amostras | {int(y_test.sum()):,} prematuros ({y_test.mean():.1%})")
print(f"Features : {XB_test.shape[1]}")

## 1. Resumo do Modelo Final

Antes de interpretar, contextualizamos os números do modelo no conjunto de teste.

In [ ]:
m   = ops["metrics"]
m05 = ops["metrics_default_threshold"]
ci  = ops["bootstrap_ci_95"]

summary = pd.DataFrame([
    {
        "Threshold": f"{m05['threshold']:.2f} (padrão)",
        "Recall":    f"{m05['recall']:.1%}",
        "Precision": f"{m05['precision']:.1%}",
        "F2":        f"{m05['f2']:.3f}",
        "ROC-AUC":   f"{m05['roc_auc']:.3f}",
        "TP / FN":   f"{m05['tp']:,} / {m05['fn']:,}",
    },
    {
        "Threshold": f"{m['threshold']:.2f} (ajustado)",
        "Recall":    f"{m['recall']:.1%}  [{ci['recall']['low']:.1%}\u2013{ci['recall']['high']:.1%}]",
        "Precision": f"{m['precision']:.1%}  [{ci['precision']['low']:.1%}\u2013{ci['precision']['high']:.1%}]",
        "F2":        f"{m['f2']:.3f}  [{ci['f2']['low']:.3f}\u2013{ci['f2']['high']:.3f}]",
        "ROC-AUC":   f"{m['roc_auc']:.3f}  [{ci['roc_auc']['low']:.3f}\u2013{ci['roc_auc']['high']:.3f}]",
        "TP / FN":   f"{m['tp']:,} / {m['fn']:,}",
    },
])

display(summary.set_index("Threshold"))

print(f"\nFP ao threshold ajustado : {m['fp']:,}  (razao FP/TP = {m['fp']/m['tp']:.1f}x)")
print(f"Brier score              : {m['brier']:.4f}  (baseline naive ~ {y_test.mean()*(1-y_test.mean()):.4f})")

## 2. Análise de Threshold

O threshold padrão (0.5) trata prematuridade como evento com probabilidade de referência 50%, subestimando positivos em datasets desbalanceados. Para triagem clínica, o objetivo é maximizar **recall** (identificar o máximo de prematuros) aceitando volume maior de falsos alarmes.

O threshold 0.40 foi escolhido em validação dedicada (20% do treino) como o ponto que maximiza F2 sujeito ao piso de recall ≥ 0.80.

In [ ]:
curve = pd.read_csv(METRICS_DIR / "threshold_search_curve.csv")

fig, ax = plt.subplots(figsize=(9, 4.5))

ax.plot(curve["threshold"], curve["recall"],    label="Recall",    color="#2196F3", lw=2)
ax.plot(curve["threshold"], curve["precision"], label="Precision", color="#FF9800", lw=2)
ax.plot(curve["threshold"], curve["f2"],        label="F2",        color="#4CAF50", lw=2, ls="--")

ax.axvline(THRESHOLD, color="red", lw=1.5, ls=":", label=f"Threshold={THRESHOLD:.2f}")
ax.axhline(0.80, color="gray", lw=1, ls="--", alpha=0.6, label="Piso recall=0.80")

row = curve[curve["threshold"].sub(THRESHOLD).abs() < 0.011].iloc[0]
for metric, color in [("recall", "#2196F3"), ("precision", "#FF9800"), ("f2", "#4CAF50")]:
    ax.annotate(
        f"{row[metric]:.2f}",
        xy=(THRESHOLD, row[metric]),
        xytext=(THRESHOLD + 0.03, row[metric]),
        color=color, fontsize=9, va="center",
    )

ax.set_xlabel("Threshold de decisao")
ax.set_ylabel("Valor da metrica")
ax.set_title("Recall, Precision e F2 por threshold (conjunto de teste)")
ax.legend(loc="upper right")
ax.set_xlim(0.05, 0.95)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_threshold_curve.png", dpi=150)
plt.show()

## 3. Importância Global de Features — SHAP

**SHAP** (SHapley Additive exPlanations) decompõe cada previsão do modelo em contribuições individuais de cada feature, garantindo consistência matemática com a teoria dos jogos cooperativos.

Para o HistGradientBoosting calibrado (3 estimadores internos via `CalibratedClassifierCV`), calculamos SHAP para cada estimador interno e fazemos a média — garantindo que a interpretação reflita o ensemble completo.

> **Como ler os gráficos SHAP:**  
> SHAP > 0 → feature empurra a previsão para **prematuro**  
> SHAP < 0 → feature empurra a previsão para **não-prematuro**  
> |SHAP| → magnitude do impacto

Usamos uma amostra de 2.000 registros do conjunto de teste para eficiência computacional. Os valores são cacheados em `results/artifacts/` para reruns rápidos.

In [ ]:
SHAP_CACHE = ARTIFACTS_DIR / "shap_values_test_2000.npz"

rng      = np.random.default_rng(RANDOM_STATE)
idx_shap = rng.choice(len(XB_test), size=SHAP_SAMPLE, replace=False)
X_shap   = XB_test.iloc[idx_shap].reset_index(drop=True)
y_shap   = y_test.iloc[idx_shap].reset_index(drop=True)

if SHAP_CACHE.exists():
    cached     = np.load(SHAP_CACHE)
    mean_shap  = cached["shap_values"]
    base_value = float(cached["base_value"])
    print(f"SHAP carregado do cache: {SHAP_CACHE.name}")
else:
    print(f"Calculando SHAP para {SHAP_SAMPLE:,} amostras (3 estimadores internos)...")
    inners    = [cc.estimator for cc in model.calibrated_classifiers_]
    shap_mats = []
    base_vals = []
    for i, inner in enumerate(inners):
        exp = shap.TreeExplainer(inner)
        sv  = exp(X_shap)
        shap_mats.append(sv.values)
        # expected_value pode ser array (um valor por classe) ou escalar
        ev = np.atleast_1d(exp.expected_value)
        base_vals.append(float(ev[-1]))  # classe positiva
        print(f"  estimador {i+1}/3 concluido")

    mean_shap  = np.mean(shap_mats, axis=0)
    base_value = float(np.mean(base_vals))
    np.savez(SHAP_CACHE, shap_values=mean_shap, base_value=base_value)
    print(f"Cache salvo: {SHAP_CACHE.name}")

shap_exp = shap.Explanation(
    values        = mean_shap,
    base_values   = base_value,
    data          = X_shap.values,
    feature_names = XB_test.columns.tolist(),
)

print(f"\nbase_value (log-odds medio do treino): {base_value:.4f}")
print(f"Shape dos valores SHAP: {mean_shap.shape}")

In [ ]:
mean_abs_shap = np.abs(mean_shap).mean(axis=0)
importance_df = (
    pd.DataFrame({"feature": XB_test.columns, "mean_abs_shap": mean_abs_shap})
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)

top15 = importance_df.head(15)

fig, ax = plt.subplots(figsize=(8, 5.5))
bars = ax.barh(
    top15["feature"][::-1],
    top15["mean_abs_shap"][::-1],
    color="#1565C0", alpha=0.85,
)
ax.set_xlabel("Importancia SHAP media (|SHAP|)")
ax.set_title("Top 15 features por impacto medio no modelo (SHAP global)")
ax.grid(True, axis="x", alpha=0.3)
for bar, val in zip(bars, top15["mean_abs_shap"][::-1]):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=8.5)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_shap_bar.png", dpi=150)
plt.show()

print("Top 5 features:")
for _, row in importance_df.head(5).iterrows():
    print(f"  {row['feature']:35s}  {row['mean_abs_shap']:.4f}")

In [ ]:
plt.figure(figsize=(9, 7))
shap.plots.beeswarm(shap_exp, max_display=15, show=False)
plt.title("SHAP Beeswarm — distribuicao do impacto por feature")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()

print("""
Leitura do beeswarm:
  Cada ponto = um registro do conjunto de avaliacao
  Cor azul   = valor baixo da feature | cor vermelho = valor alto
  Posicao X  = impacto na previsao (negativo = reduz risco / positivo = aumenta risco)
""")

## 4. Efeito Individual das Features Mais Importantes

Para as 4 features de maior impacto SHAP, analisamos **como** o valor da feature se traduz em risco previsto.

| Feature | Interpretacao esperada |
|---|---|
| `KOTELCHUCK_ORDINAL` | Mais adequado o pre-natal → menor risco de prematuridade |
| `MESPRENAT` | Inicio tardio do pre-natal → maior risco |
| `IDADEMAE` | Extremos de idade (adolescentes e > 35 anos) → maior risco (curva em U) |
| `QTDFILVIVO` | Alta paridade pode ser protetora (gestacoes anteriores bem-sucedidas) |

In [ ]:
def ordinal_to_rank(series):
    sorted_vals = sorted(series.unique())
    return series.map({v: i for i, v in enumerate(sorted_vals)})

KOTELCHUCK_LABELS = ["Sem pre-natal", "Inadequado", "Intermediario", "Adequado", "Mais que adequado"]

kotc_rank = ordinal_to_rank(X_shap["KOTELCHUCK_ORDINAL"])
kotc_col  = XB_test.columns.get_loc("KOTELCHUCK_ORDINAL")
kotc_shap_vals = mean_shap[:, kotc_col]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# KOTELCHUCK: SHAP por nivel
ax = axes[0]
groups = [kotc_shap_vals[kotc_rank == n] for n in range(5)]
bp = ax.boxplot(groups, labels=KOTELCHUCK_LABELS, patch_artist=True,
                medianprops={"color": "red", "lw": 2})
for patch, color in zip(bp["boxes"], ["#B71C1C","#E53935","#FB8C00","#43A047","#1B5E20"]):
    patch.set_facecolor(color)
    patch.set_alpha(0.65)
ax.axhline(0, color="gray", lw=1, ls="--")
ax.set_title("SHAP x KOTELCHUCK_ORDINAL")
ax.set_xlabel("Adequacao do pre-natal")
ax.set_ylabel("Valor SHAP")
ax.tick_params(axis="x", rotation=20)
ax.grid(True, axis="y", alpha=0.3)

# MESPRENAT: SHAP x valor padronizado
ax2 = axes[1]
mes_col  = XB_test.columns.get_loc("MESPRENAT")
sc = ax2.scatter(
    X_shap["MESPRENAT"].values, mean_shap[:, mes_col],
    c=y_shap.values, cmap="coolwarm", alpha=0.35, s=15, linewidths=0,
)
ax2.axhline(0, color="gray", lw=1, ls="--")
ax2.set_title("SHAP x MESPRENAT (inicio do pre-natal)")
ax2.set_xlabel("MESPRENAT padronizado (maior = mais tardio)")
ax2.set_ylabel("Valor SHAP")
plt.colorbar(sc, ax=ax2, label="PREMATURO (0/1)")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_shap_dependence_kotc_mesprenat.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# IDADEMAE
ax = axes[0]
ida_col = XB_test.columns.get_loc("IDADEMAE")
sc = ax.scatter(
    X_shap["IDADEMAE"].values, mean_shap[:, ida_col],
    c=y_shap.values, cmap="coolwarm", alpha=0.35, s=15, linewidths=0,
)
ax.axhline(0, color="gray", lw=1, ls="--")
ax.set_title("SHAP x IDADEMAE (idade da mae)")
ax.set_xlabel("IDADEMAE padronizado (menor = mais jovem)")
ax.set_ylabel("Valor SHAP")
plt.colorbar(sc, ax=ax, label="PREMATURO (0/1)")
ax.grid(True, alpha=0.3)

# QTDFILVIVO
ax2 = axes[1]
qtd_col = XB_test.columns.get_loc("QTDFILVIVO")
sc2 = ax2.scatter(
    X_shap["QTDFILVIVO"].values, mean_shap[:, qtd_col],
    c=y_shap.values, cmap="coolwarm", alpha=0.35, s=15, linewidths=0,
)
ax2.axhline(0, color="gray", lw=1, ls="--")
ax2.set_title("SHAP x QTDFILVIVO (numero de filhos vivos)")
ax2.set_xlabel("QTDFILVIVO padronizado")
ax2.set_ylabel("Valor SHAP")
plt.colorbar(sc2, ax=ax2, label="PREMATURO (0/1)")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_shap_dependence_idade_filhos.png", dpi=150)
plt.show()

## 5. Interpretação Clínica dos Resultados SHAP

### O que o modelo aprendeu

**`KOTELCHUCK_ORDINAL` — dominância do pré-natal**

A adequação do pré-natal (índice de Kotelchuck) é a feature mais importante por larga margem — ~2.5x o impacto da segunda colocada. O índice captura tanto a quantidade quanto a oportunidade das consultas em relação à idade gestacional esperada. Gestantes sem pré-natal ou com atendimento inadequado recebem SHAP positivo alto → modelo as classifica com alto risco de prematuridade.

**`MESPRENAT` — momento de entrada no sistema**

Início tardio do pré-natal (≥ 4º mês) correlaciona-se com SHAP positivo. O sinal é parcialmente independente de Kotelchuck: uma gestante pode ter iniciado tarde mas completado muitas consultas — MESPRENAT captura especificamente o atraso na entrada, que reduz a janela de intervenção precoce.

**`IDADEMAE` — curva em U não-linear**

Mães muito jovens (< 17 anos) e mais velhas (> 35 anos) têm SHAP positivo — padrão U documentado na literatura obstétrica. Os mecanismos diferem: imaturidade fisiológica nas adolescentes; comorbidades crônicas e complicações vasculares nas mais velhas. O HistGB capturou essa não-linearidade que modelos lineares perderiam.

**`QTDFILVIVO` — história obstétrica de sucesso**

O número de filhos vivos funciona como um indicador de **paridade bem-sucedida anterior**. O SHAP mostra sinal predominantemente negativo (protetor) para valores altos — gestantes que já tiveram filhos vivos têm evidência de capacidade de levar gestações a termo. Primíparas (QTDFILVIVO = 0) tendem a SHAP positivo leve, pois não existe histórico obstétrico para ancorar o risco.

> **Atenção à leitura:** QTDFILVIVO não é diretamente sobre "ter muitos filhos". O que o modelo está capturando é *gestações anteriores sem perda fetal registrada* como sinal protetor. Uma gestante com 3 filhos vivos e 0 perdas tem perfil muito diferente de uma com 3 filhos vivos e 2 perdas — e o HISTPERDAFETAL (que permanece no cenário A) capturaria a segunda distinção. No cenário B, QTDFILVIVO absorve parte desse sinal de forma mais grosseira.

**`LONGITUDE` — desigualdade regional**

Municípios mais a leste (região Nordeste/litoral) apresentam maior risco previsto. O modelo aprendeu essa assimetria regional como proxy de acesso precário à saúde e vulnerabilidade socioeconômica, sem que ela fosse explicitamente codificada como variável de região.

## 5.1 Embasamento Científico — SHAP vs. Literatura

### Estamos no teto do SINASC?

A resposta curta é **sim**. Weber et al. (2018) avaliaram modelos de ML em certidões de nascimento administrativas da Califórnia (N = 336 mil) — o mais próximo metodologicamente do SINASC — e reportaram AUC entre **0.62 e 0.67** usando apenas dados administrativos, sem medidas clínicas. Nosso modelo atingiu **AUC = 0.653**, exatamente dentro desse intervalo.

| Estudo | Tipo de dado | AUC | Nota |
|---|---|---|---|
| Weber et al. (2018) *Ann Epidemiol* | Certidão de nascimento + alta hospitalar (EUA) | 0.62–0.67 | Administrativo puro, sem clínica |
| **Nosso modelo** | **SINASC (Brasil)** | **0.653** | **Administrativo puro** |
| Rocha et al. (2021) *Lancet Reg Health Am* | SINASC + PMAQ + censo (Brasil) | RMSE = 2.09 sem. | Regressão, sem AUC direto |
| Koivu & Sairanen (2022) *BMC Medicine* | Códigos de faturamento (EHR) | ~0.65–0.75 | Administrativo com CID |
| Kong et al. (2024) *Front Pediatrics* | Alta hospitalar + CID (China) | 0.82–0.85 | Administrativo com diagnósticos |
| Sun et al. (2022) *J Healthcare Eng* | Prontuário completo com exames | **0.885** | Dados clínicos: labs, sinais vitais |

**Conclusão do benchmarking:** O salto de 0.65 → 0.88 que a literatura reporta ao adicionar dados clínicos (comprimento cervical, fibronectina fetal, exames laboratoriais) não é possível com o SINASC, pois esses campos não existem no sistema. Chegamos ao limite de informação do dado administrativo.

---

### As features mais importantes são o que a ciência indica?

Sim — e a correspondência é notável:

| Posição SHAP | Feature do modelo | Embasamento científico | Referência |
|---|---|---|---|
| 🥇 1 | `KOTELCHUCK_ORDINAL` | Pré-natal inadequado OR = 1.29–3.98 para prematuridade espontânea | Leal et al. (2016); Aguiar et al. (2023) *PLOS Global PH* |
| 🥈 2 | `MESPRENAT` | OMS recomenda início antes da 12ª semana; início tardio reduz janela de intervenção | WHO ANC Guidelines (2016) |
| 🥉 3 | `IDADEMAE` | Relação em U confirmada: aOR = 1.08 (20–24 anos) e 1.20 (≥40 anos) vs. 30–34 anos | Fuchs et al. (2018) *PLOS ONE*, N = 165 mil |
| 4 | `QTDFILVIVO` | Paridade alta como fator protetor em gestações anteriores bem-sucedidas | Goldenberg et al. (2008) *Lancet* |
| 5 | `LONGITUDE` | Disparidades regionais no Brasil — Nordeste tem maior prevalência e menor acesso | Leal et al. (2016); CIDACS Cohort (Paixão 2021) |

> **Referência seminal:** Goldenberg RL, Culhane JF, Iams JD, Romero R. *"Epidemiology and causes of preterm birth."* Lancet, 371(9606):75–84, 2008. — Identifica que os preditores administrativos mais relevantes são exatamente os que capturamos: assistência pré-natal, idade materna e paridade. Os preditores clínicos mais fortes (comprimento cervical, fibronectina) simplesmente não existem no SINASC.

---

### ⚠️ Aviso sobre causalidade reversa no KOTELCHUCK

O índice de Kotelchuck foi desenhado para populações a termo. Quando uma gestação termina prematuramente na semana 32, o denominador do índice (número esperado de consultas) é calculado para uma gestação completa — classificando mecanicamente a gestante como "atendimento inadequado", mesmo que tenha recebido cuidado apropriado por 32 semanas.

Bloch et al. (2009) e Shin & Song (2019) documentam esse efeito: **KOTELCHUCK_ORDINAL pode ser parcialmente uma consequência da prematuridade, não apenas uma causa**. O modelo captura um sinal estatístico válido, mas a interpretação causal exige cautela. Isso não invalida o uso para triagem — o sinal existe e é clinicamente relevante — mas impede inferência causal direta.

**Referências:**
- Bloch et al. (2009). *"Application of the Kessner and Kotelchuck prenatal care adequacy indices in a preterm birth population."* Public Health Nursing, 26(5):449–459.
- Shin D & Song WO. (2019). *"Influence of the APNCU Index on Small-For-Gestational-Age and Preterm Births."* J Clin Med, 8(7):978.
- Kotelchuck M. (1994). *"An evaluation of the Kessner Adequacy of Prenatal Care Index..."* Am J Public Health, 84(9):1414–1420.

## 6. Explicacao Individual — SHAP Local

O grafico *waterfall* decompoe a previsao para **um unico registro** em contribuicoes aditivas de cada feature:

- **E[f(X)]** — valor de base: previsao media do modelo em todo o dataset de treino
- **Barras vermelhas** — features que aumentam o risco em relacao ao valor de base
- **Barras azuis** — features que reduzem o risco
- **f(x)** — previsao final: soma do valor de base com todas as contribuicoes

In [ ]:
y_prob_shap = model.predict_proba(X_shap)[:, 1]

# Alto risco: maior prob prevista entre os verdadeiros prematuros (TP)
tp_idx   = np.where(y_shap.values == 1)[0]
idx_high = tp_idx[np.argmax(y_prob_shap[tp_idx])]

# Baixo risco: menor prob prevista entre os verdadeiros nao-prematuros (TN)
tn_idx  = np.where(y_shap.values == 0)[0]
idx_low = tn_idx[np.argmin(y_prob_shap[tn_idx])]

print(f"Caso alto risco  — prob={y_prob_shap[idx_high]:.3f} | real=PREMATURO")
print(f"Caso baixo risco — prob={y_prob_shap[idx_low]:.3f}  | real=NAO PREMATURO")

In [ ]:
plt.figure(figsize=(9, 6))
shap.plots.waterfall(shap_exp[idx_high], max_display=12, show=False)
plt.title(f"SHAP Waterfall — caso alto risco (prob={y_prob_shap[idx_high]:.3f}, real=PREMATURO)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_shap_waterfall_high.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(9, 6))
shap.plots.waterfall(shap_exp[idx_low], max_display=12, show=False)
plt.title(f"SHAP Waterfall — caso baixo risco (prob={y_prob_shap[idx_low]:.3f}, real=NAO PREMATURO)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_shap_waterfall_low.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Analise Estratificada — Equidade do Modelo

Um modelo com boa performance global pode ter desempenho desigual entre grupos populacionais. Verificamos recall, precision e F2 por:

- **Raca da mae** — proxy de desigualdade racial no acesso a saude
- **Escolaridade** — proxy socioeconomico
- **Estado civil** — proxy de suporte social

> Recall desigual entre grupos significa que o modelo identifica prematuros de alguns grupos melhor do que de outros — impacto direto em quem recebe atencao clinica adicional.

In [ ]:
def stratified_metrics(y_true, y_prob, groups, threshold, min_size=200, min_pos=30):
    y_pred = (y_prob >= threshold).astype(int)
    rows = []
    for grp in sorted(groups.unique()):
        mask = groups == grp
        if mask.sum() < min_size or y_true[mask].sum() < min_pos:
            continue
        rows.append({
            "Grupo":      grp,
            "N":          int(mask.sum()),
            "Prematuros": int(y_true[mask].sum()),
            "Prev.": f"{y_true[mask].mean():.1%}",
            "Recall":     recall_score(y_true[mask], y_pred[mask], zero_division=0),
            "Precision":  precision_score(y_true[mask], y_pred[mask], zero_division=0),
            "F2":         fbeta_score(y_true[mask], y_pred[mask], beta=2, zero_division=0),
        })
    return pd.DataFrame(rows).set_index("Grupo")

def recon_dummies(df, prefix):
    cols = [c for c in df.columns if c.startswith(prefix + "_")]
    return df[cols].idxmax(axis=1).str.replace(prefix + "_", "", regex=False)

def ordinal_to_label(series, labels):
    return series.map({v: labels[i] for i, v in enumerate(sorted(series.unique()))})

raca      = recon_dummies(XB_test, "RACACORMAE")
civ       = recon_dummies(XB_test, "ESTCIVMAE")
esc_label = ordinal_to_label(
    XB_test["ESCMAE2010_ORDINAL"],
    ["0-Sem escolaridade", "1-Fund.I incompleto", "2-Fund. completo",
     "3-Medio incompleto", "4-Medio completo", "5-Superior"],
)

In [ ]:
fmt = {"Recall": "{:.1%}", "Precision": "{:.1%}", "F2": "{:.3f}"}

print("RECALL POR RACA DA MAE")
df_raca = stratified_metrics(y_test.values, y_prob, raca, THRESHOLD)
display(df_raca.style.format(fmt).highlight_min(subset=["Recall"], color="#FFCDD2").highlight_max(subset=["Recall"], color="#C8E6C9"))

print("\nRECALL POR ESCOLARIDADE DA MAE")
df_esc = stratified_metrics(y_test.values, y_prob, esc_label, THRESHOLD)
display(df_esc.style.format(fmt).highlight_min(subset=["Recall"], color="#FFCDD2").highlight_max(subset=["Recall"], color="#C8E6C9"))

print("\nRECALL POR ESTADO CIVIL")
df_civ = stratified_metrics(y_test.values, y_prob, civ, THRESHOLD)
display(df_civ.style.format(fmt).highlight_min(subset=["Recall"], color="#FFCDD2").highlight_max(subset=["Recall"], color="#C8E6C9"))

In [ ]:
global_recall = ops["metrics"]["recall"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
datasets = [
    (df_raca, "Raca da mae"),
    (df_esc,  "Escolaridade"),
    (df_civ,  "Estado civil"),
]

for ax, (df, title) in zip(axes, datasets):
    recalls = df["Recall"].values
    groups  = df.index.tolist()
    colors  = ["#1976D2" if r >= 0.75 else "#E53935" for r in recalls]
    bars    = ax.barh(groups, recalls, color=colors, alpha=0.8)
    ax.axvline(global_recall, color="black", lw=1.5, ls="--", label=f"Global ({global_recall:.1%})")
    for bar, val in zip(bars, recalls):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                f"{val:.1%}", va="center", fontsize=8)
    ax.set_xlim(0, 1.05)
    ax.set_xlabel("Recall")
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, axis="x", alpha=0.3)

plt.suptitle("Recall por grupo demografico — threshold = 0.40", fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_stratified_recall.png", dpi=150)
plt.show()

## 8. Calibracao do Modelo

Um modelo **calibrado** produz probabilidades confiaveis: se o modelo preve 30% de chance de prematuridade, aproximadamente 30% dos casos com essa previsao devem ser de fato prematuros.

O `CalibratedClassifierCV` com metodo *sigmoid* foi aplicado justamente para isso. Avaliamos a qualidade da calibracao no conjunto de teste via *reliability diagram*.

In [ ]:
frac_pos, mean_pred_val = calibration_curve(y_test, y_prob, n_bins=10, strategy="quantile")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Calibracao perfeita")
ax.plot(mean_pred_val, frac_pos, "o-", color="#1565C0", lw=2, ms=7, label="Modelo")
ax.fill_between(mean_pred_val, frac_pos, mean_pred_val, alpha=0.12, color="#1565C0")
ax.set_xlabel("Probabilidade prevista")
ax.set_ylabel("Fracao real de positivos")
ax.set_title("Reliability diagram")
ax.legend()
ax.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.hist(y_prob[y_test == 0], bins=50, alpha=0.6, color="#42A5F5",
         density=True, label="Nao prematuro")
ax2.hist(y_prob[y_test == 1], bins=50, alpha=0.6, color="#EF5350",
         density=True, label="Prematuro")
ax2.axvline(THRESHOLD, color="black", lw=1.5, ls=":", label=f"Threshold={THRESHOLD:.2f}")
ax2.set_xlabel("Probabilidade prevista")
ax2.set_ylabel("Densidade")
ax2.set_title("Distribuicao de probabilidades por classe")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_calibration.png", dpi=150)
plt.show()

baseline_brier = float(y_test.mean() * (1 - y_test.mean()))
brier          = ops["metrics"]["brier"]
print(f"Brier Score do modelo  : {brier:.4f}")
print(f"Brier Score naive      : {baseline_brier:.4f}")
print(f"Reducao relativa       : {(baseline_brier - brier)/baseline_brier:.1%}")

## 9. Aplicabilidade Prática no SUS e Fontes de Dados Complementares

### O modelo poderia ser usado no SUS hoje?

**Sim — como ferramenta de triagem na atenção básica, não como diagnóstico.**

Viellas et al. (2020) mostram que a cobertura pré-natal no SUS é quase universal (> 95% das gestantes têm ao menos uma consulta), mas a qualidade cai para cerca de 25% quando critérios mínimos de adequação são aplicados. O problema não é acesso — é priorização. É exatamente aí que um modelo de triagem tem valor: **identificar, dentro da massa de gestantes atendidas, quais precisam de atenção especializada (pré-natal de alto risco)**.

Rocha et al. (2021) publicaram no *Lancet Regional Health – Americas* um modelo brasileiro com dados SINASC + PMAQ explicitamente voltado para o SUS, concluindo que "a disponibilidade de dados administrativos do SUS oferece oportunidade única para desenvolver um algoritmo de estratificação de risco com dados mínimos de cada paciente."

#### Fluxo de operação proposto

```
Registro da gestante no SINASC/e-SUS APS
        ↓
Modelo calcula probabilidade de prematuridade
(dados disponíveis na abertura do pré-natal: idade, paridade, escolaridade,
 estado civil, município, mês da 1ª consulta)
        ↓
Prob ≥ 0.40  →  Sinalizar para pré-natal de ALTO RISCO
                (encaminhamento a maternidade de referência,
                 frequência aumentada de consultas, exames adicionais)
        ↓
Prob < 0.40  →  Pré-natal de RISCO HABITUAL na UBS
```

**Impacto estimado na prática:** ao threshold 0.40, o modelo sinaliza ~63% do total de gestantes como "alto risco" (TP + FP juntos). Isso é alto para um encaminhamento direto, mas viável como *priorização de atenção adicional* — não necessariamente encaminhamento a serviço especializado. Uma variação com threshold mais alto (ex: 0.60) produziria menor carga com menor recall.

---

### Com quais outras fontes de dados poderíamos melhorar?

A infraestrutura de linkage já existe no Brasil: o **CIDACS (Centro de Integração de Dados para Saúde)** vincula 24,6 milhões de nascimentos ao Cadastro Único, SIM, SIHSUS e SINAN (Paixão et al., 2021, *Int J Epidemiol*). Cada fonte adicional endereça uma limitação diferente do SINASC:

| Fonte | O que adiciona | Limitação atual do SINASC que resolve | Ganho esperado de AUC |
|---|---|---|---|
| **e-SUS APS** | Visitas individuais de pré-natal, PA, peso, glicemia, altura uterina, diagnósticos CID | SINASC só tem contagem agregada de consultas | Alto (+0.05–0.10) |
| **SIHSUS** | Internações durante a gestação (pré-eclâmpsia, ITU, hemorragia), histórico de prematuridade anterior | Histórico de prematuridade anterior — o **maior preditor individual** (OR = 3.74, Leal 2016) — não está no SINASC | Alto (+0.05–0.08) |
| **Cadastro Único (CadÚnico)** | Renda, vulnerabilidade social, composição familiar, acesso a saneamento | Proxy socioeconômico atual: apenas escolaridade e estado civil | Moderado (+0.02–0.04) |
| **SINAN** | Infecções notificáveis durante gestação (sífilis, HIV, dengue, zika) | Infecções são causas diretas de prematuridade mas invisíveis no SINASC | Moderado (+0.02–0.05) |
| **CNES** | Distância e qualidade do estabelecimento de referência obstétrica | Longitude/latitude são proxies grosseiros de acesso | Leve (+0.01–0.02) |
| **Prontuário eletrônico / EHR** | Comprimento cervical, fibronectina fetal, exames laboratoriais completos | Ausentes totalmente do sistema público | Muito alto (+0.15–0.23) |

> **A adição mais impactante seria o histórico de prematuridade anterior** (disponível no SIHSUS). Leal et al. (2016) encontraram OR = 3.74 — nenhuma outra variável disponível no SINASC chega perto. Rocha et al. (2022, *BMC Medicine*) demonstraram que essa variável muda substancialmente o perfil de risco de recorrência quando resgatada via linkage SINASC+SIHSUS.

#### Trajetória realista de melhoria

```
SINASC apenas (hoje)         → AUC ≈ 0.65  [teto confirmado]
+ SIHSUS (histórico PTB)     → AUC ≈ 0.70–0.72
+ e-SUS APS (visitas clínicas) → AUC ≈ 0.74–0.77
+ CadÚnico + SINAN           → AUC ≈ 0.76–0.80
+ Prontuário (EHR completo)  → AUC ≈ 0.85–0.90  [teto clínico]
```

Estimativas baseadas nos benchmarks de Koivu & Sairanen (2022), Kong et al. (2024) e Sun et al. (2022).

---

**Referências desta seção:**
- Rocha et al. (2021). *"Data-driven risk stratification for preterm birth in Brazil."* Lancet Reg Health Am, 8:100169.
- Viellas EF et al. (2020). *"Prenatal care in the Brazilian public health services."* Rev Saúde Pública, 54:08.
- Paixão ES et al. (2021). *"CIDACS Birth Cohort Profile."* Int J Epidemiol, 50(1):37–38.
- Rocha et al. (2022). *"Differences in risk factors for incident and recurrent preterm birth: CIDACS cohort."* BMC Medicine, 20:146.
- Leal MC et al. (2016). *"Prevalence and risk factors related to preterm birth in Brazil."* Reprod Health, 13(Suppl 3).
- Koivu & Sairanen (2022). *"Dense phenotyping from EHR enables ML-based prediction of preterm birth."* BMC Medicine, 20:428.
- Kong et al. (2024). *"Predicting preterm birth using auto-ML frameworks."* Front Pediatrics, 12:1330420.
- Sun et al. (2022). *"ML-Based Prediction Model of Preterm Birth Using EHR."* J Healthcare Eng. PMC9020923.

## 9. Limitacoes e Conclusoes

### O que o modelo nao sabe

| Limitacao | Impacto |
|---|---|
| **Correlacoes individuais fracas (< 0.10)** | O modelo esta proximo do teto do dataset. ROC-AUC 0.653 e o maximo alcancavel com estas features. |
| **Precision de ~12%** | Para cada prematuro identificado, ~7 nao-prematuros sao sinalizados. Aceitavel para triagem populacional; inaceitavel para diagnostico individual. |
| **Ausencia de features clinicas chave** | Comorbidades maternas (hipertensao, diabetes, incompetencia cervical), historico de pre-eclampsia e exames laboratoriais nao estao no SINASC. |
| **Gap CV AP x Test AP** | CV Average Precision (0.664) e inflado pelo `sample_weight=balanced` no treino. Test AP (0.206) e o numero confiavel. |
| **Generalizacao temporal** | Treinado em 2020-2022 (inclui periodo COVID). Drift de distribuicao possivel em dados pos-2022. |
| **Nao causalidade** | SHAP mede associacao, nao causalidade. Um Kotelchuck alto pode ser indicador de acesso a saude, nao causa direta de menor risco. |

### O que o modelo entrega

- **Identificacao de ~81% dos prematuros** (recall = 0.813) com threshold = 0.40 usando apenas dados administrativos do SINASC
- **Triagem escalavel**: processa 138 mil registros de teste sem custo marginal
- **Prioridade clinica clara**: gestantes com Kotelchuck baixo + inicio tardio de pre-natal sao as de maior risco previsto
- **Instrumento de equidade**: analise estratificada permite identificar grupos com recall sistematicamente menor e direcionar politicas publicas

### Proximos passos

1. **Validacao prospectiva** em dados de 2023-2024
2. **Linkage com prontuarios** para adicionar comorbidades maternas
3. **Retreinamento estratificado** se analise de equidade revelar recall < 70% em algum grupo
4. **Monitoramento de drift** usando `results/metrics/retraining_checklist.json` como referencia